# 1. Case Definition && Business Understanding

## 1.1 Problem Definition

Case Scenario: You are working as a Senior Data Scientist at a Klavi a firm specialized in credit risk. A Fintech Client wants to review their credit granting policy and needs to define:

 Which variables are most relevant for predicting default

 Which variables are good candidates for a practical credit policy (rules + score)

 How to turn these variables into an objective and justifiable proposal for the business team


 Challenge: You are given one week to analyze the data files provided by the client. You must deliver a recommendation on how to proceed using Klavi’s variables. Using a visual approach, show how to analyze the data and extract meaningful business insights.

Please detail:
 The main steps you would take to explore, clean, and organize the provided dataset.
 The key factors or criteria you would consider essential to ensure the insights are relevant, accurate, and representative.
 How you would validate both the analytical approach and the insights generated.

Please fell free to contact Andrea Filgueiras at andrea.filgueiras@klavi.ai if you have any questions

## 1.2 Objective of this study

The objetive of this notebook is provide evidences to give an anwser to the questions below:

1. Which variables are most relevant for predicting default?

2. Which variables are good candidates for a pratical credit policy (rules + score)?

3. How to turn these variables into an objective and justifiable proposal for the business team?

Challenge: You are given one week to analyze the data files provided by the client. You must deliver a recommendation on how to proceed using Klavi’s variables. Using a visual approach, show how to analyze the data and extract meaningful business insights.

In other words, the objective of this analysis is to identify the most relevant variables for predicting credit default and to propose a practical and explainable credit policy. The final solution must support business decision-making by translating data insights into actionable rules and/or a scoring system.


## 1.3 Business Understanding

At this point, default (target variable), can be provided in the dataset as a binary variable. It is importante to understand and validated:
- The time horizon of default (for example, 30, 60, 90 days past due);
- If there are variables that contain information related to the post-default period (data leakage risk);
- The business meaning of default (rules).

This validation will be performed during the data exploration phase.

The proposed solution will consider:
- Interpretability: variables must be explainable to business stakeholders;
- Simplicity: rules should be implementable in production systems;
- Stability: variables must be robust over time;
- Regulatory compliance: avoid non-transparent or biased variables.

The success of the solution will be evaluated based on:
- Predictive power (for example, AUC, KS, separation between good/bad);
- Business interpretability;
- Ease of implementation;
- Alignment with business objectives;
- Robustness and stability.

# 2. Import Libs && DataBase

Pandas and NumPy will be used to manipulate datasets and perform calculations. Path from pathlib will help us manage file paths in a more flexible way.

In [137]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'seaborn'

Two parquet files were provided as the data source. We will now read these files and start exploring them.

In [3]:
base_profile = pd.read_parquet("./profile_case_credit.parquet")

base_transactions = pd.read_parquet("./transactions_case_credit.parquet")

FileNotFoundError: [Errno 2] No such file or directory: './profile_case_credit.parquet'

# 3. EDA - Explore Data Analysis - Data Undertanding

## 3.1 Data Undertanding 

Let's start with transactions dataset

### 3.1.1 DataBase transactions

#### 3.1.1.1 Let’s start by looking at the columns

In [5]:
base_transactions.columns

Index(['uuid', 'REFERENCIA', 'score_klavi_1', 'score_klavi_2', 'classe_social',
       'status_empregaticio', 'regiao', 'Renda', 'risco_aposta',
       'Capacidade Financeira'],
      dtype='str')

This dataset have 10 variables.

In [17]:
base_transactions.info()

<class 'pandas.DataFrame'>
Index: 371596 entries, 0 to 1198093
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   uuid                   371596 non-null  str    
 1   REFERENCIA             371596 non-null  str    
 2   score_klavi_1          32277 non-null   float64
 3   score_klavi_2          331573 non-null  float64
 4   classe_social          371596 non-null  str    
 5   status_empregaticio    371596 non-null  str    
 6   regiao                 371013 non-null  str    
 7   Renda                  32277 non-null   float64
 8   risco_aposta           32277 non-null   float64
 9   Capacidade Financeira  325286 non-null  float64
dtypes: float64(5), str(5)
memory usage: 52.7 MB


there is missing values

In [40]:
transactions_quality = pd.DataFrame({
    "missing_data": base_transactions.isnull().mean()
})

print("\nTransactions missing values:\n", transactions_quality)


Transactions missing values:
                        missing_data
uuid                       0.000000
referencia                 0.000000
score_klavi_1              0.913140
score_klavi_2              0.107706
classe_social              0.000000
status_empregaticio        0.000000
regiao                     0.001569
renda                      0.913140
risco_aposta               0.913140
capacidade financeira      0.124625


- 371596 lines x 10 columns

Based on the column names, REFERENCIA appears to be related to transaction dates, although it is not in the correct datetime format. There are 5 continuous numerical variables (score_klavi_1, score_klavi_2, renda, risco_aposta, and capacidade_financeira) and 3 categorical variables (classe_social, status_empregaticio, and regiao). The uuid and REFERENCIA columns act as identifiers to track clients over time.

Additionally, only uuid, REFERENCIA, classe_social, and status_empregaticio do not contain missing values. Others have.

In [25]:
base_transactions.head(10)

,uuid,REFERENCIA,score_klavi_1,score_klavi_2,classe_social,status_empregaticio,regiao,Renda,risco_aposta,Capacidade Financeira
0,7b0c9b36fe0f730f7c0d081534ff9a2b,2024-06-12,58.07,512.0,D/E,Desconhecido,Sudeste,1052.65,0.0,839.64
3,513b46119c7d33fc324f3c561905c159,2025-02-06,NaN,422.0,C,Desconhecido,Sudeste,NaN,NaN,2213.37
7,0d7fed4a5b92cbae24e2b80e49fb0a4e,2025-06-13,NaN,607.0,C,Desconhecido,Sudeste,NaN,NaN,587.04
8,c158eeca94b81f5f63cee1496dbea081,2024-07-30,NaN,NaN,C,Autônomo,Sudeste,NaN,NaN,1961.59
9,97ebff4ac03d546fcb03040d1866bea0,2024-11-04,NaN,533.0,B,Desconhecido,Sudeste,NaN,NaN,3585.16
17,66ef8c1ba69e1b72cca182ac9aa94971,2025-01-27,NaN,527.0,C,Desconhecido,Sudeste,NaN,NaN,1072.74
20,c66eb83f48ed6af14fea6181df023f29,2024-08-13,NaN,432.0,D/E,Desconhecido,Sudeste,NaN,NaN,NaN
21,d2261a8d3ff96ed3ff70351d4ff0b1f4,2024-09-24,NaN,555.0,D/E,-999999,Sudeste,NaN,NaN,NaN
23,d87779754701ad49498a532889d0af92,2024-12-07,NaN,476.0,D/E,Autônomo,Sudeste,NaN,NaN,923.51
25,9809cebb519c0d12215a5609be07c73c,2025-02-08,NaN,598.0,B,Autônomo,Sudeste,NaN,NaN,2316.47


In [4]:
base_transactions.describe()

,score_klavi_1,score_klavi_2,Renda,risco_aposta,Capacidade Financeira
count,32277.000000,331573.000000,32277.000000,32277.000000,3.252860e+05
mean,-72475.672167,520.963981,2850.452457,1.412585,1.925911e+03
std,259379.346501,60.696365,3600.016877,2.739815,5.038808e+03
min,-999999.000000,345.000000,317.280000,0.000000,-1.088145e+06
25%,40.340000,481.000000,1321.570000,0.000000,8.038300e+02
50%,57.250000,524.000000,2066.940000,0.000000,1.470810e+03
75%,68.190000,566.000000,3057.770000,1.000000,2.361590e+03
max,100.000000,663.000000,129296.190000,10.000000,1.207959e+05


#### 3.1.1.2 Apparently, score_klavi_1 is a float variable with decimal values, while score_klavi_2 does not contain decimal values. Let's test:

In [15]:
test_integer = base_transactions["score_klavi_2"]
test_integer = test_integer.dropna()

In [18]:
(test_integer % 1 == 0).all()

np.True_

Explanation:
- test_integer % 1 computes the remainder of each value when divided by 1. If a number is an integer (example, 2.0), the remainder is 0;
- The comparison == 0 returns a boolean series indicating which values are integers;
- .all() returns True only if all values satisfy this condition.

Interpretation:
- True -> all values are integer-like;
- False -> at least one value has a decimal component.

The result is .true, so score_klavi_2 can be consider as integer while score_klavi_1 is float.

In [35]:
test_integer_risco = base_transactions["risco_aposta"]
test_integer_risco = test_integer_risco.dropna()
(test_integer_risco % 1 == 0).all()

np.True_

same test for risk column. Is a categorical variable, with integer values.

In [ ]:
test_integer_risco = base_transactions["risco_aposta"]
test_integer_risco = test_integer_risco.dropna()
(test_integer_risco % 1 == 0).all()

#### 3.1.1.3 Let’s examine how uuid is stored. In the transactions dataset, by counting the distinct pairs of uuid and REFERENCIA, we can understand how the samples are structured.

In [ ]:
base_transactions.groupby(['uuid', 'REFERENCIA']).size().reset_index(name='counts').describe()

,counts
count,371596.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


This code evaluates the uniqueness and distribution of records in the dataset by counting how many times each combination of uuid and REFERENCIA appears, and then summarizes those counts using descriptive statistics.

The min and max values are both 1, which means that for combinations uuid + REFERENCIA there are no duplicates.

### 3.1.2 DataBase profile

#### 3.1.2.1 For database profile, let's follow the same structure of transaction analysis.

In [4]:
base_profile.columns

Index(['uuid', 'DATA', 'SAFRA', 'OVER30M2', 'STATUS'], dtype='str')

5 variables

In [18]:
base_profile.info()

<class 'pandas.DataFrame'>
RangeIndex: 220618 entries, 0 to 220617
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   uuid      220618 non-null  str    
 1   DATA      220618 non-null  str    
 2   SAFRA     220618 non-null  float64
 3   OVER30M2  24148 non-null   float64
 4   STATUS    220618 non-null  str    
dtypes: float64(2), str(3)
memory usage: 19.1 MB


220.618 lines x 5 columns

In [41]:
profile_quality = pd.DataFrame({
    "missing_data": base_profile.isnull().mean()
})

print("Profile missing values:\n", profile_quality)

Profile missing values:
           missing_data
uuid          0.000000
DATA          0.000000
SAFRA         0.000000
OVER30M2      0.890544
STATUS        0.000000


In [26]:
24148 / 220618 

0.10945616404826441

The date column is currently in string format, so it needs to be converted to a datetime type. This will allow us to perform date-related operations and analyses. Same case of transaction dataset.

The safra column can also be treated as a datetime variable.

OVER30M2 can be our TARGET. The OVER30M2 variable contains a large number of missing values. Only 24.148 rows have valid values, while 196.470 are missing. This means that only about 11% of the dataset contains values for the potential target variable.

At first glance, there is one numeric variable (safra, although it refers to a date), one date variable (in the format), and two categorical variables (over30m2, apparently the target, and status)

Lets see the first 10 lines of this dataset

In [28]:
base_profile.head(10)

,uuid,DATA,SAFRA,OVER30M2,STATUS
0,d946dabe8351c29c3ca9db3c7005341b,2024-08-24,202408.0,0.0,Contratado
1,b9aa3d3552e2e21f92d34df2540de5c3,2024-09-02,202409.0,1.0,Contratado
2,ac5371c706da6f7a57c9b9593a8a5d04,2024-11-25,202411.0,0.0,Contratado
3,d44880bed90eaacb2fba3feead83ca16,2024-09-09,202409.0,0.0,Contratado
4,8ec0eefeda7ae58d5a98a2d59aca4bbb,2024-06-18,202406.0,0.0,Contratado
5,f678146b3eca8e6712135fddc35fb3ba,2024-09-07,202409.0,0.0,Contratado
6,9dea08ff1d666272f3c60e56e3398876,2024-11-19,202411.0,0.0,Contratado
7,862fdc1869d84d8adbae80677c99b7b3,2025-05-28,202505.0,0.0,Contratado
8,010b18ce36ae05c53de14917e90191b3,2025-05-08,202505.0,0.0,Contratado
9,bbb827213dcec48e8725392cf0423273,2025-06-20,202506.0,0.0,Contratado


In [3]:
base_profile.describe()

,SAFRA,OVER30M2
count,220618.000000,24148.000000
mean,202450.283331,0.103031
std,46.821529,0.304006
min,202406.000000,0.000000
25%,202409.000000,0.000000
50%,202412.000000,0.000000
75%,202503.000000,0.000000
max,202506.000000,1.000000


#### 3.1.2.2 Understanding OVER30M2

Let's see the nulls in OVER30M2

In [ ]:
base_profile[base_profile["OVER30M2"].isnull()]

,uuid,DATA,SAFRA,OVER30M2,STATUS
24148,ad1a7008e6f8f7d6fd2d0f4261afc6f6,2024-08-30,202408.0,NaN,Aprovado
24149,4a84c5458fa12c70961366a21153ea5f,2024-11-28,202411.0,NaN,Aprovado
24150,483aceaad439624ead11b2d5ca1e5af9,2025-03-25,202503.0,NaN,Aprovado
24151,b08c6ef9df0ab742179d73b6a1cb0e54,2024-07-27,202407.0,NaN,Aprovado
24152,ec6bf5147fa3da1ae8f0bb74f3c9953e,2024-10-22,202410.0,NaN,Aprovado
...,...,...,...,...,...
220613,ff0bdb200e964e3370f1bc28449518a2,2025-04-14,202504.0,NaN,Reprovado
220614,3b7d62731c1edba4556ae538f846388c,2024-07-27,202407.0,NaN,Reprovado
220615,0a2e0aa37e2bcbb715ef1a82ab20b4b3,2025-01-22,202501.0,NaN,Reprovado
220616,df8d2d77a231c93559ef22b3889dab4a,2024-11-09,202411.0,NaN,Reprovado


Lets understand the distribution of the values

In [3]:
quantidade_over30m2_null = base_profile[base_profile["OVER30M2"].isnull()].shape[0]
quantidade_over30m2_1 = base_profile[base_profile["OVER30M2"] == 1].shape[0]
quantidade_over30m2_0 = base_profile[base_profile["OVER30M2"] == 0].shape[0]

print(f"Number of lines with OVER30M2 null: {quantidade_over30m2_null}, that represents {quantidade_over30m2_null / base_profile.shape[0] * 100:.2f}% of total.")
print(f"Number of lines with OVER30M2 equal to 1: {quantidade_over30m2_1}, that represents {quantidade_over30m2_1 / base_profile.shape[0] * 100:.2f}% of total.")
print(f"Number of lines with OVER30M2 equal to 0: {quantidade_over30m2_0}, that represents {quantidade_over30m2_0 / base_profile.shape[0] * 100:.2f}% of total.") 

Number of lines with OVER30M2 null: 196470, that represents 89.05% of total.
Number of lines with OVER30M2 equal to 1: 2488, that represents 1.13% of total.
Number of lines with OVER30M2 equal to 0: 21660, that represents 9.82% of total.


These missing values can be not a random and likely represent observations for which the performance window has not yet been completed (for example, right-censored data).

To ensure a valid supervised learning setup, its apropriate that observations with known outcomes (OVER30M2 = 0 or 1) be considered for modeling.

This results in a reduced but reliable dataset where the target variable is properly defined.

#### 3.1.2.3 Understanding date, safra and uuid

First, to better understand the date variable, i will cast it to a datetime format to ensure proper analysis and correct handling by Python.

In [5]:
base_profile['DATA'] = pd.to_datetime(base_profile['DATA'], errors='coerce')

In [7]:
base_profile.info()

<class 'pandas.DataFrame'>
RangeIndex: 220618 entries, 0 to 220617
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   uuid      220618 non-null  str           
 1   DATA      220618 non-null  datetime64[us]
 2   SAFRA     220618 non-null  float64       
 3   OVER30M2  24148 non-null   float64       
 4   STATUS    220618 non-null  str           
dtypes: datetime64[us](1), float64(2), str(2)
memory usage: 17.0 MB


pd.to_datetime(...):
- Attempts to convert the column into a datetime type (datetime64[ns]), which is the standard format for handling dates in pandas.

base_profile['DATA']:
- The column being converted, which likely contains date information stored as strings or mixed formats.

errors='coerce'
- If a value cannot be converted into a valid date, it will be replaced with NaT (Not a Time), instead of raising an error

In [6]:
base_profile['DATA'].min(), base_profile['DATA'].max()

(Timestamp('2024-06-01 00:00:00'), Timestamp('2025-06-30 00:00:00'))

The dataset spans a one-year period, from June 2024 to June 2025.

In [14]:
base_profile["SAFRA_DATE"] = pd.to_datetime(
    base_profile["SAFRA"].astype(int).astype(str),
    format="%Y%m"
)

base_profile.head(5)

,uuid,DATA,SAFRA,OVER30M2,STATUS,SAFRA_DATE
0,d946dabe8351c29c3ca9db3c7005341b,2024-08-24,202408.0,0.0,Contratado,2024-08-01
1,b9aa3d3552e2e21f92d34df2540de5c3,2024-09-02,202409.0,1.0,Contratado,2024-09-01
2,ac5371c706da6f7a57c9b9593a8a5d04,2024-11-25,202411.0,0.0,Contratado,2024-11-01
3,d44880bed90eaacb2fba3feead83ca16,2024-09-09,202409.0,0.0,Contratado,2024-09-01
4,8ec0eefeda7ae58d5a98a2d59aca4bbb,2024-06-18,202406.0,0.0,Contratado,2024-06-01


In [15]:
base_profile['SAFRA_DATE'].min(), base_profile['SAFRA_DATE'].max()

(Timestamp('2024-06-01 00:00:00'), Timestamp('2025-06-01 00:00:00'))

In the safra column, the time span is the same as in the date column

In [16]:
base_profile.groupby(['uuid', 'SAFRA_DATE']).size().reset_index(name='counts').describe()

,SAFRA_DATE,counts
count,209838,209838.000000
mean,2024-11-27 03:47:14.563806,1.051373
min,2024-06-01 00:00:00,1.000000
25%,2024-09-01 00:00:00,1.000000
50%,2024-12-01 00:00:00,1.000000
75%,2025-03-01 00:00:00,1.000000
max,2025-06-01 00:00:00,8.000000
std,NaN,0.285252


In [17]:
base_profile.groupby(['uuid', 'DATA']).size().reset_index(name='counts').describe()

,counts
count,220618.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


There are no duplicates for the uuid + date combination, while duplicates exist for the uuid + safra combination. This indicates that, in the profile dataset, customer information is captured more than once per month. It was observed that the maximum number of repetitions is 8 within a single month.

As a consequence, SAFRA should not be treated as the unit of analysis, and care must be taken to avoid multiple observations of the same customer when building the analytical dataset. To ensure consistency and avoid bias, a single snapshot per customer can be selected, representing the most recent information available prior to the target observation. Or we can consider as something near to a daily data set.

## 3.2 Conclusion

The exploratory data analysis provided important insights into the structure, quality, and limitations of the datasets, guiding the next steps of data preparation and modeling.

---

### Data Structure and Granularity

The analysis revealed that the datasets operate at different levels of granularity:

- The `profile` dataset is time-dependent, with multiple observations per customer across different dates.
- The `transactions` dataset contains customer-level attributes with a temporal reference (`REFERENCIA`).

It was observed that:
- The combination of `uuid + DATA` is unique in the profile dataset
- Multiple records exist for the same customer within a given month (`SAFRA`)

This indicates that the data is captured at a finer temporal resolution (likely daily or event-based), requiring careful alignment between features and target to avoid data leakage.

---

### Target Variable Considerations

The target variable (`OVER30M2`) presents a significant proportion of missing values (~89%), which can be not random and likely represent right-censored observations (cases where the performance window has not yet been completed).

To ensure a valid supervised learning setup:
- Only records with non-null target values can be considered
- The analytical dataset will be constructed using observed outcomes only

Additionally, the target distribution among valid observations shows a typical credit risk imbalance, with a relatively low default rate, reinforcing the need for appropriate evaluation metrics.

---

### Data Quality Observations

Several data quality aspects were identified:

- The `SAFRA` variable was stored as a float and required type correction before being converted into a proper datetime format
- Date fields required standardization to ensure proper temporal analysis
- No duplication was found at the expected primary key levels (`uuid + DATA`, `uuid + REFERENCIA`), confirming data integrity
- Some variables may require further inspection regarding missing values and consistency

These issues will be addressed in the Bronze layer to ensure a clean and reliable dataset.

---

### Feature Considerations

The dataset includes a mix of:

- Internal scoring variables (score_klavi_1, score_klavi_2)
- Socioeconomic attributes (Renda, classe_social)
- Behavioral indicators (risco_aposta, Capacidade Financeira)

Initial inspection suggests that:

- Score variables are strong candidates for predictive modeling and policy design
- Financial capacity and income-related variables are likely relevant for risk segmentation
- Some variables may require transformation (binning) to improve interpretability and usability in policy rules

The distinction between continuous and discrete variables was also evaluated, which will guide feature engineering decisions in later stages.

---

### Next Steps

Based on the EDA findings, the following steps will be implemented:

#### Bronze Layer
- Data type corrections (for example, SAFRA and DATA)
- Standardization of date formats
- Handling of missing and inconsistent values
- Filtering of valid target observations (OVER30M2 not null)

#### Silver Layer
- Definition of the analytical population (one record per customer)
- Temporal alignment between features and target
- Feature engineering and transformation
- Selection of candidate variables for analysis

#### Gold Layer
- Construction of the final analytical dataset
- Consolidation of features and target
- Preparation for business analysis and policy design

---

### Final Considerations

The EDA phase highlighted the importance of temporal consistency, target definition, and data quality in building a reliable credit risk analysis.

Rather than directly applying modeling techniques, the focus will be on constructing a well-defined analytical dataset and extracting meaningful, business-oriented insights to support credit policy decisions.

# 4. Bronze Layer Construction and Analysis

definition of paths to datasets

In [106]:
BASE_PATH = Path("data")

RAW_PATH = BASE_PATH / "raw_data"
BRONZE_PATH = BASE_PATH / "bronze_data"

loading data

In [107]:
base_transactions = pd.read_parquet(RAW_PATH / "transactions_case_credit.parquet")
base_profile = pd.read_parquet(RAW_PATH / "profile_case_credit.parquet")

## 4.1 Profile DataSet Transformations

In [41]:
# Convert DATA to datetime
base_profile["DATA"] = pd.to_datetime(
    base_profile["DATA"],
    errors="coerce"
)

# Convert SAFRA (YYYYMM float/int → datetime)
base_profile["SAFRA"] = pd.to_datetime(
    base_profile["SAFRA"].astype("Int64").astype(str),
    format="%Y%m",
    errors="coerce"
)

# Standardize column names (optional but recommended)
base_profile.columns = base_profile.columns.str.lower()

# Save the transformed dataset in the bronze layer
base_profile.to_parquet(
    BRONZE_PATH / "profile_bronze.parquet",
    index=False
)

In [42]:
base_profile.info()

<class 'pandas.DataFrame'>
RangeIndex: 220618 entries, 0 to 220617
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   uuid      220618 non-null  str           
 1   data      220618 non-null  datetime64[us]
 2   safra     220618 non-null  datetime64[us]
 3   over30m2  24148 non-null   float64       
 4   status    220618 non-null  str           
dtypes: datetime64[us](2), float64(1), str(2)
memory usage: 17.0 MB


In [43]:
base_profile.head(10)

,uuid,data,safra,over30m2,status
0,d946dabe8351c29c3ca9db3c7005341b,2024-08-24,2024-08-01,0.0,Contratado
1,b9aa3d3552e2e21f92d34df2540de5c3,2024-09-02,2024-09-01,1.0,Contratado
2,ac5371c706da6f7a57c9b9593a8a5d04,2024-11-25,2024-11-01,0.0,Contratado
3,d44880bed90eaacb2fba3feead83ca16,2024-09-09,2024-09-01,0.0,Contratado
4,8ec0eefeda7ae58d5a98a2d59aca4bbb,2024-06-18,2024-06-01,0.0,Contratado
5,f678146b3eca8e6712135fddc35fb3ba,2024-09-07,2024-09-01,0.0,Contratado
6,9dea08ff1d666272f3c60e56e3398876,2024-11-19,2024-11-01,0.0,Contratado
7,862fdc1869d84d8adbae80677c99b7b3,2025-05-28,2025-05-01,0.0,Contratado
8,010b18ce36ae05c53de14917e90191b3,2025-05-08,2025-05-01,0.0,Contratado
9,bbb827213dcec48e8725392cf0423273,2025-06-20,2025-06-01,0.0,Contratado


## 4.2 Transactions Dataset Transformations

In [44]:
# Convert REFERENCIA to datetime
base_transactions["REFERENCIA"] = pd.to_datetime(
    base_transactions["REFERENCIA"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Convert score_klavi_2 to integer
base_transactions["score_klavi_2"] = pd.to_numeric(
    base_transactions["score_klavi_2"],
     errors="coerce"
).astype("Int64")

# Convert risco_aposta to integer
base_transactions["risco_aposta"] = pd.to_numeric(
    base_transactions["risco_aposta"],
     errors="coerce"
).astype("Int64")

# Standardize column names
base_transactions.columns = base_transactions.columns.str.lower()

base_transactions.to_parquet(
    BRONZE_PATH / "transactions_bronze.parquet",
    index=False
)

In [45]:
base_transactions.info()

<class 'pandas.DataFrame'>
Index: 371596 entries, 0 to 1198093
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   uuid                   371596 non-null  str           
 1   referencia             371596 non-null  datetime64[us]
 2   score_klavi_1          32277 non-null   float64       
 3   score_klavi_2          331573 non-null  Int64         
 4   classe_social          371596 non-null  str           
 5   status_empregaticio    371596 non-null  str           
 6   regiao                 371013 non-null  str           
 7   renda                  32277 non-null   float64       
 8   risco_aposta           32277 non-null   Int64         
 9   capacidade financeira  325286 non-null  float64       
dtypes: Int64(2), datetime64[us](1), float64(3), str(4)
memory usage: 49.9 MB


In [46]:
base_transactions.head(10)

,uuid,referencia,score_klavi_1,score_klavi_2,classe_social,status_empregaticio,regiao,renda,risco_aposta,capacidade financeira
0,7b0c9b36fe0f730f7c0d081534ff9a2b,2024-06-12,58.07,512,D/E,Desconhecido,Sudeste,1052.65,0,839.64
3,513b46119c7d33fc324f3c561905c159,2025-02-06,NaN,422,C,Desconhecido,Sudeste,NaN,<NA>,2213.37
7,0d7fed4a5b92cbae24e2b80e49fb0a4e,2025-06-13,NaN,607,C,Desconhecido,Sudeste,NaN,<NA>,587.04
8,c158eeca94b81f5f63cee1496dbea081,2024-07-30,NaN,<NA>,C,Autônomo,Sudeste,NaN,<NA>,1961.59
9,97ebff4ac03d546fcb03040d1866bea0,2024-11-04,NaN,533,B,Desconhecido,Sudeste,NaN,<NA>,3585.16
17,66ef8c1ba69e1b72cca182ac9aa94971,2025-01-27,NaN,527,C,Desconhecido,Sudeste,NaN,<NA>,1072.74
20,c66eb83f48ed6af14fea6181df023f29,2024-08-13,NaN,432,D/E,Desconhecido,Sudeste,NaN,<NA>,NaN
21,d2261a8d3ff96ed3ff70351d4ff0b1f4,2024-09-24,NaN,555,D/E,-999999,Sudeste,NaN,<NA>,NaN
23,d87779754701ad49498a532889d0af92,2024-12-07,NaN,476,D/E,Autônomo,Sudeste,NaN,<NA>,923.51
25,9809cebb519c0d12215a5609be07c73c,2025-02-08,NaN,598,B,Autônomo,Sudeste,NaN,<NA>,2316.47


# 5. Silver Layer Construction and Analysis

Start with path and load data that we want to manipulate.

In [120]:
BASE_PATH = Path("data")

BRONZE_PATH = BASE_PATH / "bronze_data"
SILVER_PATH = BASE_PATH / "silver_data"

In [121]:
bronze_profile = pd.read_parquet(BRONZE_PATH / "profile_bronze.parquet")
bronze_transactions = pd.read_parquet(BRONZE_PATH / "transactions_bronze.parquet")

## 5.1 To construct the silver layer, let’s start by defining the target population: valid profiles with a known target

### 5.1.1 lets filter only target not null

In [110]:
profile_valid = bronze_profile[bronze_profile["over30m2"].notnull()]
profile_valid.info()

<class 'pandas.DataFrame'>
RangeIndex: 24148 entries, 0 to 24147
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   uuid      24148 non-null  str           
 1   data      24148 non-null  datetime64[us]
 2   safra     24148 non-null  datetime64[us]
 3   over30m2  24148 non-null  float64       
 4   status    24148 non-null  str           
dtypes: datetime64[us](2), float64(1), str(2)
memory usage: 1.9 MB


Now, we have the public.

1 line per cliente?

In [111]:
profile_valid["uuid"].nunique(), profile_valid.shape[0]

(23299, 24148)

there is duplicated uuid because is a daily information.

## 5.2 Now we have 24.148 uuid + data as spine to merge with the transactions dataset. These combinations will form our first table in the silver layer, called silver_spine.

In [113]:
# Save the transformed dataset in the bronze layer
silver_spine = profile_valid[["uuid", "data"]]
silver_spine.to_parquet(
    SILVER_PATH / "spine_silver.parquet",
    index=False
)

In [114]:
silver_spine.info()

<class 'pandas.DataFrame'>
RangeIndex: 24148 entries, 0 to 24147
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   uuid    24148 non-null  str           
 1   data    24148 non-null  datetime64[us]
dtypes: datetime64[us](1), str(1)
memory usage: 1.1 MB


## 5.3 We can now separate the transaction features and the profile features in the silver layer 

In [122]:
silver_feat_transaction = bronze_transactions
silver_feat_transaction["data"] = bronze_transactions["referencia"]
silver_feat_profile = profile_valid.drop(columns=["safra"])

In [124]:
silver_feat_transaction.to_parquet(
    SILVER_PATH / "silver_feat_transaction.parquet",
    index=False
)

silver_feat_profile.to_parquet(
    SILVER_PATH / "silver_feat_profile.parquet",
    index=False
)

# 6. Gold Layer Construction and Analysis

In [125]:
BASE_PATH = Path("data")

SILVER_PATH = BASE_PATH / "silver_data"
GOLD_PATH = BASE_PATH / "gold_data"

In [126]:
silver_spine = pd.read_parquet(SILVER_PATH / "spine_silver.parquet")
silver_transactions = pd.read_parquet(SILVER_PATH / "silver_feat_transaction.parquet")
silver_profile = pd.read_parquet(SILVER_PATH / "silver_feat_profile.parquet")

In [129]:
first_df_gold = silver_spine.merge(
    silver_transactions,
    on=["uuid", "data"],
    how="left",
    suffixes=("", "_trans")
)

In [130]:
silver_spine.info()

<class 'pandas.DataFrame'>
RangeIndex: 24148 entries, 0 to 24147
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   uuid    24148 non-null  str           
 1   data    24148 non-null  datetime64[us]
dtypes: datetime64[us](1), str(1)
memory usage: 1.1 MB


In [131]:
first_df_gold.info()

<class 'pandas.DataFrame'>
RangeIndex: 24148 entries, 0 to 24147
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   uuid                   24148 non-null  str           
 1   data                   24148 non-null  datetime64[us]
 2   referencia             12442 non-null  datetime64[us]
 3   score_klavi_1          1081 non-null   float64       
 4   score_klavi_2          11089 non-null  Int64         
 5   classe_social          12442 non-null  str           
 6   status_empregaticio    12442 non-null  str           
 7   regiao                 12415 non-null  str           
 8   renda                  1081 non-null   float64       
 9   risco_aposta           1081 non-null   Int64         
 10  capacidade financeira  11008 non-null  float64       
dtypes: Int64(2), datetime64[us](2), float64(3), str(4)
memory usage: 3.0 MB


In [132]:
df_gold = first_df_gold.merge(
    silver_profile,
    on=["uuid", "data"],
    how="left",
    suffixes=("", "_profile")
)

In [133]:
df_gold.info()

<class 'pandas.DataFrame'>
RangeIndex: 24148 entries, 0 to 24147
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   uuid                   24148 non-null  str           
 1   data                   24148 non-null  datetime64[us]
 2   referencia             12442 non-null  datetime64[us]
 3   score_klavi_1          1081 non-null   float64       
 4   score_klavi_2          11089 non-null  Int64         
 5   classe_social          12442 non-null  str           
 6   status_empregaticio    12442 non-null  str           
 7   regiao                 12415 non-null  str           
 8   renda                  1081 non-null   float64       
 9   risco_aposta           1081 non-null   Int64         
 10  capacidade financeira  11008 non-null  float64       
 11  over30m2               24148 non-null  float64       
 12  status                 24148 non-null  str           
dtypes: Int64(2),

In [134]:
# renomear a coluna capacidade financeira para capacidade_financeira
df_gold = df_gold.rename(columns={"capacidade financeira": "capacidade_financeira"})
df_gold.info()

<class 'pandas.DataFrame'>
RangeIndex: 24148 entries, 0 to 24147
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   uuid                   24148 non-null  str           
 1   data                   24148 non-null  datetime64[us]
 2   referencia             12442 non-null  datetime64[us]
 3   score_klavi_1          1081 non-null   float64       
 4   score_klavi_2          11089 non-null  Int64         
 5   classe_social          12442 non-null  str           
 6   status_empregaticio    12442 non-null  str           
 7   regiao                 12415 non-null  str           
 8   renda                  1081 non-null   float64       
 9   risco_aposta           1081 non-null   Int64         
 10  capacidade_financeira  11008 non-null  float64       
 11  over30m2               24148 non-null  float64       
 12  status                 24148 non-null  str           
dtypes: Int64(2),

In [ ]:
df_gold.to_parquet(
    GOLD_PATH / "df_gold.parquet",
    index=False
)

In [136]:
print("Shape:", df_gold.shape)
print("\nTarget distribution:")
print(df_gold["over30m2"].value_counts(normalize=True))

print("\nMissing values:")
print(df_gold.isnull().mean().sort_values(ascending=False).head(10))

Shape: (24148, 13)

Target distribution:
over30m2
0.0    0.896969
1.0    0.103031
Name: proportion, dtype: float64

Missing values:
score_klavi_1            0.955234
risco_aposta             0.955234
renda                    0.955234
capacidade_financeira    0.544144
score_klavi_2            0.540790
regiao                   0.485879
classe_social            0.484761
referencia               0.484761
status_empregaticio      0.484761
data                     0.000000
dtype: float64
